Bibliotecas utilizadas

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

Leitura do arquivo

In [ ]:
data = pd.read_csv('../data/olist_orders_dataset.csv')

print(data.head())

In [ ]:
data.info()

Transformação das colunas de string para datetime

In [ ]:
colunasdt_data = ['order_purchase_timestamp',
                'order_approved_at',
                'order_delivered_carrier_date',
                'order_delivered_customer_date',
                'order_estimated_delivery_date'
                ]
data[colunasdt_data] = data[colunasdt_data].apply(pd.to_datetime)

print(data.info())

Criando 2 colunas, sendo:
Uma para calcular o prazo desde o envio da entrega, até o recebimento do cliente.
Outra para calcular o prazo da aprovação do pedido, para o envio da entrega. 

In [ ]:
data['prazo_entrega_dias'] = (data['order_delivered_customer_date'] - data['order_delivered_carrier_date']).dt.days
data['prazo_saida_dias'] = (data['order_delivered_carrier_date'] - data['order_approved_at']).dt.days

print(data.head())

E criando a média dessas duas colunas, para identificar quais são as médias do prazo de entrega ao cliente, e do prazo de envio após a aprovação da compra.

In [ ]:
media_prazo_entrega = data['prazo_entrega_dias'].mean()
media_prazo_saida = data['prazo_saida_dias'].mean()

print(f"Média do prazo de entrega: {media_prazo_entrega:.1f} dias")
print(f"Média do prazo de saída: {media_prazo_saida:.1f} dias")

Criei uma coluna com a comparação real entre a data prevista e a data de entrega.

In [ ]:
data['estimada_real'] = (data['order_estimated_delivery_date'] - data['order_delivered_customer_date']).dt.days
print(data.head(20)[['estimada_real', 'order_estimated_delivery_date', 'order_delivered_customer_date']])

Também criei uma coluna com a média da diferença entre a data prevista e a data de entrega.

In [ ]:
media_estimada_real = data['estimada_real'].mean()
print(f"Média da diferença entre a data estimada e a data real de entrega: {media_estimada_real:.1f} dias")

Criei uma variável para identificar a quantidade de atrasos

In [ ]:
atraso = data[data['estimada_real'] < 0]
print(f"Quantidade de entregas atrasadas: {len(atraso)}")

E uma variável para identificar o tempo médio de atraso.

In [ ]:
media_atraso = atraso['estimada_real'].mean()
print(f"Média do atraso das entregas: {media_atraso:.1f} dias")

Usei função .value_counts() para identificar os valores diferentes na coluna 'order_status'.

In [ ]:
order_status_counts = data['order_status'].value_counts()
print(order_status_counts)


E utilizei somente o valor 'delivered' para posteriormente comparar os atrasos somente com as compras que foram de fato entregues.

In [ ]:
order_delivered = data[data['order_status'] == 'delivered']
print(f"Quantidade de entregas concluídas: {len(order_delivered)}")

Comparação:

In [ ]:
delivered_atraso = order_delivered[order_delivered['estimada_real'] < 0]
delivered_sem_atraso = order_delivered[order_delivered['estimada_real'] >= 0]
print(f"Quantidade de entregas concluídas com atraso: {len(delivered_atraso)}")
print(f"Quantidade de entregas concluídas sem atraso: {len(delivered_sem_atraso)}")

Por fim, utilizei um gráfico de pizza para exibir a proporção de entregas atrasadas e no prazo.

In [ ]:
plt.pie([len(delivered_atraso), len(delivered_sem_atraso)], labels=['Atrasadas', 'No Prazo'], autopct='%1.1f%%')
plt.title('Proporção de Entregas Atrasadas e no Prazo')
plt.show()